In [42]:
#import yfinance as yf
#data = yf.download("BTC-USD",start= "2020-01-01" , end = "2026-9-15")
#data.to_csv("BTC-USD.csv")

In [43]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import classification_report , accuracy_score , confusion_matrix
from sklearn.model_selection import GridSearchCV , TimeSeriesSplit , train_test_split

In [44]:
df=pd.read_csv("BTC-USD.csv")
df=pd.DataFrame(df)
print(type(df))
print(df.head(10))


<class 'pandas.DataFrame'>
         Date        Close         High          Low         Open       Volume
0  2020-01-01  7200.174316  7254.330566  7174.944336  7194.892090  18565664997
1  2020-01-02  6985.470215  7212.155273  6935.270020  7202.551270  20802083465
2  2020-01-03  7344.884277  7413.715332  6914.996094  6984.428711  28111481032
3  2020-01-04  7410.656738  7427.385742  7309.514160  7345.375488  18444271275
4  2020-01-05  7411.317383  7544.497070  7400.535645  7410.451660  19725074095
5  2020-01-06  7769.219238  7781.867188  7409.292969  7410.452148  23276261598
6  2020-01-07  8163.692383  8178.215820  7768.227539  7768.682129  28767291327
7  2020-01-08  8079.862793  8396.738281  7956.774414  8161.935547  31672559265
8  2020-01-09  7879.071289  8082.295898  7842.403809  8082.295898  24045990466
9  2020-01-10  8166.554199  8166.554199  7726.774902  7878.307617  28714583844


In [45]:
df.info()
print(df.describe())
df["Date"]=pd.to_datetime(df["Date"])
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2449 entries, 0 to 2448
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    2449 non-null   str    
 1   Close   2449 non-null   float64
 2   High    2449 non-null   float64
 3   Low     2449 non-null   float64
 4   Open    2449 non-null   float64
 5   Volume  2449 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 114.9 KB
               Close           High            Low           Open  \
count    2449.000000    2449.000000    2449.000000    2449.000000   
mean    49851.970717   50774.951990   48820.290259   49824.645441   
std     30858.990451   31319.666575   30371.519134   30865.783448   
min      4970.788086    5331.833984    4106.980957    5002.578125   
25%     23783.029297   24209.660156   23245.021484   23774.648438   
50%     43798.117188   44408.664062   42786.917969   43792.019531   
75%     68518.093750   69867.351562   67194.882812   68512.179688 

In [46]:
df["benefit"]=df["Close"]-df["Open"]

df["Return"] = df["Close"].pct_change()

df["HL_range"] = (df["High"] - df["Low"]) / df["Close"]

df["OC_range"] = (df["Close"] - df["Open"]) / df["Open"]

df["Volume_Change"] = df["Volume"].pct_change()

df["Tomarrow"] = df["Close"].shift(-1)

#print(df.isna().sum())
df = df.dropna()
print(df.isna().sum())

df["Target"] = (df["Tomarrow"] > df["Close"]).astype(int)
print(df.head(10))


Date             0
Close            0
High             0
Low              0
Open             0
Volume           0
benefit          0
Return           0
HL_range         0
OC_range         0
Volume_Change    0
Tomarrow         0
dtype: int64
         Date        Close         High          Low         Open  \
1  2020-01-02  6985.470215  7212.155273  6935.270020  7202.551270   
2  2020-01-03  7344.884277  7413.715332  6914.996094  6984.428711   
3  2020-01-04  7410.656738  7427.385742  7309.514160  7345.375488   
4  2020-01-05  7411.317383  7544.497070  7400.535645  7410.451660   
5  2020-01-06  7769.219238  7781.867188  7409.292969  7410.452148   
6  2020-01-07  8163.692383  8178.215820  7768.227539  7768.682129   
7  2020-01-08  8079.862793  8396.738281  7956.774414  8161.935547   
8  2020-01-09  7879.071289  8082.295898  7842.403809  8082.295898   
9  2020-01-10  8166.554199  8166.554199  7726.774902  7878.307617   
10 2020-01-11  8037.537598  8218.359375  8029.642090  8162.190918   


In [47]:
X = df[["Open", "High", "Low", "Close", "Volume", "benefit" ,"Return","HL_range" , "OC_range", "Volume_Change"]]

#X = df[["Return","HL_range" , "OC_range", "Volume_Change"]]


y = df["Target"]

#Train on whole of data 

In [48]:
clf = XGBClassifier()
tscv = TimeSeriesSplit(n_splits=5)

param = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}


grid = GridSearchCV(
    estimator=clf,
    param_grid=param,
    cv= tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)
print(grid.best_params_)
best_model = grid.best_estimator_
print(f"Best estimator = {best_model}")

y_pred = best_model.predict(X)

print("Accuracy:", accuracy_score(y, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y, y_pred))

print("Classification Report:")
print(classification_report(y, y_pred))


Fitting 5 folds for each of 32 candidates, totalling 160 fits
{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
Best estimator = XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)
Accuracy: 0.8941561095218635
Confusion Matrix:
[[1078  128]
 [

#Train and Test Split

In [49]:
X_train, X_test, y_train , y_test = train_test_split(X , y ,test_size=0.1, shuffle = False , random_state=42)
print(f"X_Train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")



X_Train shape: (2202, 10)
X_test shape: (245, 10)
y_train shape: (2202,)
y_test shape: (245,)


#XGBoost on train and test data

In [50]:
clf = XGBClassifier(random_state=42 , eval_metric="logloss" , n_jobs=-1)
tscv = TimeSeriesSplit(n_splits=5)

param = {
    "n_estimators": [200 , 300],
    "max_depth": [3, 5 ],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.5, 0.7],
    "colsample_bytree": [0.8, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}


grid = GridSearchCV(
    estimator=clf,
    param_grid=param,
    cv= tscv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print(grid.best_params_)
best_model = grid.best_estimator_
print(f"Best estimator = {best_model}")

y_pred = best_model.predict(X_train)


print("Accuracy:", accuracy_score(y_train, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_train, y_pred))

print("Classification Report:")
print(classification_report(y_train, y_pred))

#-------test part------#

y_pred_test = best_model.predict(X_test)


print("Accuracy_Test:", accuracy_score(y_test, y_pred_test))

print("Confusion Matrix_Test:")
print(confusion_matrix(y_test, y_pred_test))

print("Classification Report_ Test:")
print(classification_report(y_test, y_pred_test))



Fitting 5 folds for each of 384 candidates, totalling 1920 fits
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.5}
Best estimator = XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)
Accuracy: 